# Thu Thập Dữ Liệu Thời Tiết Từ Open-Meteo
### Data Collection · Ho Chi Minh City · Hourly Weather Data

---

> **Pipeline tổng quát:** Cấu hình tham số → Gọi Archive API → Xử lý JSON → Lưu CSV

---

## Mục Tiêu

| Hạng mục | Nội dung |
|---|---|
| **Bài toán** | Thu thập dữ liệu thời tiết theo giờ tại TP.HCM từ Open-Meteo Archive API |
| **Phạm vi địa lý** | Trung tâm TP.HCM (10.7769°N, 106.7009°E) |
| **Nguồn dữ liệu** | Open-Meteo Archive API (archive-api.open-meteo.com) |
| **Thời gian thu thập** | Từ 19/11/2024 đến 10/05/2026 |
| **Output** | `weather_openmeteo_2024_2026.csv` — dữ liệu hourly |

---

## Cấu Trúc Notebook

| Bước | Nội dung |
|---|---|
| **1** | Cấu hình API và tham số |
| **2** | Gọi Open-Meteo Archive API |
| **3** | Xử lý và chuyển đổi dữ liệu JSON → DataFrame |
| **4** | Lưu CSV và kiểm tra kết quả |

---

## Biến Thời Tiết Thu Thập

| Biến | Đơn vị | Mô tả |
|---|---|---|
| `temperature` | °C | Nhiệt độ tại độ cao 2m |
| `humidity` | % | Độ ẩm tương đối tại 2m |
| `wind_speed` | km/h | Tốc độ gió tại độ cao 10m |
| `pressure` | hPa | Áp suất bề mặt |
| `precipitation` | mm | Lượng mưa |

---

## Lưu Ý Quan Trọng

1. **Không cần API Key**: Open-Meteo Archive API miễn phí và không yêu cầu xác thực.
2. **Timezone**: Dữ liệu trả về theo `Asia/Ho_Chi_Minh` (UTC+7).
3. **Missing Values**: Open-Meteo trả về `null` cho các giờ thiếu dữ liệu — sẽ xuất hiện là `NaN` trong DataFrame.
4. **Giới hạn thời gian**: Archive API chỉ hỗ trợ dữ liệu đến 5–7 ngày trước ngày hiện tại.


In [3]:
import requests
import pandas as pd
import os

print("Gọi Open-Meteo archive API...")

LAT, LON = 10.7769, 106.7009
start_date = "2024-11-19"
end_date   = "2026-05-10"

url = (
    f"https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={LAT}&longitude={LON}"
    f"&start_date={start_date}&end_date={end_date}"
    f"&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,"
    f"surface_pressure,precipitation"
    f"&timezone=Asia%2FHo_Chi_Minh"
)

r = requests.get(url, timeout=30)
if r.status_code != 200:
    raise ConnectionError(f"API lỗi {r.status_code}: {r.text[:200]}")

w = r.json()["hourly"]
df_weather = pd.DataFrame({
    'time':          pd.to_datetime(w['time']),
    'temperature':   w['temperature_2m'],
    'humidity':      w['relative_humidity_2m'],
    'wind_speed':    w['wind_speed_10m'],
    'pressure':      w['surface_pressure'],
    'precipitation': w['precipitation'],
})
print(f"→ Dữ liệu khí tượng: {len(df_weather):,} giờ | {start_date} → {end_date}")
save_path_weather = '../data/raw/raw_openmeteo.csv'
os.makedirs(os.path.dirname(save_path_weather), exist_ok=True)
df_weather.to_csv(save_path_weather, index=False)
print(f"→ Dữ liệu khí tượng đã lưu: {save_path_weather}")


Gọi Open-Meteo archive API...
→ Dữ liệu khí tượng: 12,912 giờ | 2024-11-19 → 2026-05-10
→ Dữ liệu khí tượng đã lưu: ../data/raw/raw_openmeteo.csv
